In [1]:
!pip install -q trl
!pip install --upgrade -q pyarrow pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 56.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 163.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, b

In [2]:
# Importing the necessary libraries

import torch
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoConfig
from transformers import AutoTokenizer
import os
from torch.utils.data import Dataset, DataLoader, random_split, RandomSampler, SequentialSampler
import pandas as pd
import torch.nn.utils.parametrize as parametrize

import random
import numpy as np

import torch
from torch.utils.data import Dataset, DataLoader, random_split, RandomSampler, SequentialSampler

seed_val = 42

random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)

from transformers import get_linear_schedule_with_warmup
import datetime

import time

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B-Base"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    attn_implementation="sdpa", # <--- This is the built-in PyTorch way
    torch_dtype="auto",
    device_map="auto"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [4]:
original_model = model.cop

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [11]:
model_copy = type(model)(model.config) # get a new instance

In [12]:
model_copy.load_state_dict(model.state_dict()) # copy weights and stuff


<All keys matched successfully>

In [13]:
??model_copy.generate

In [ ]:
for target_module in target_names:


In [30]:
for name, module in model.named_modules():
  print(module)

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [20]:
extract_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
target_names = []
for name, module in model.named_modules():
  if any([True if module_name in name else False for module_name in extract_modules]):
    target_names.append(module)

In [28]:
total_paramaters_effected = 0
for index, layer in enumerate(target_names):

  total_paramaters_effected += layer.weight.nelement()


In [29]:
total_paramaters_effected

176160768

In [49]:
class LoRAParametrization(nn.Module):

  def __init__(self, features_in, features_out, rank, alpha, device):
    super().__init__()
    std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
    self.lora_A = nn.Parameter(torch.zeros((rank, features_out), device = device, dtype = torch.bfloat16))
    self.lora_B = nn.Parameter(torch.zeros((features_in, rank), device = device, dtype = torch.bfloat16))
    nn.init.normal_(self.lora_A, mean = 0, std = std_dev)
    self.scale = alpha / rank
    self.enabled = True
  def forward(self, original_weights):
    if self.enabled:
      return original_weights + (self.lora_B @ self.lora_A).view(original_weights.shape) * self.scale
    else:
      return original_weights

In [50]:
def linear_layer_parameterization(layer, device, rank = 1, lora_alpha = 1):
  features_in, features_out = layer.weight.shape

  return LoRAParametrization(features_in, features_out, rank = rank, alpha = lora_alpha, device = device)


In [51]:
device = "cuda"

In [52]:
for target_module in target_names:
  parametrize.register_parametrization(target_module, "weight", linear_layer_parameterization(target_module, device, 16, 32))

In [55]:
total_parameters_lora = 0
total_parameters_non_lora = 0
for index, layer in enumerate(target_names):
    total_parameters_lora += layer.parametrizations["weight"][0].lora_A.nelement() + layer.parametrizations["weight"][0].lora_B.nelement()
    total_parameters_non_lora += layer.weight.nelement()
    print(
        f'Layer {index+1}: W: {layer.weight.shape} + Lora_A: {layer.parametrizations["weight"][0].lora_A.shape} + Lora_B: {layer.parametrizations["weight"][0].lora_B.shape}'
    )

Layer 1: W: torch.Size([2048, 1024]) + Lora_A: torch.Size([16, 1024]) + Lora_B: torch.Size([2048, 16])
Layer 2: W: torch.Size([1024, 1024]) + Lora_A: torch.Size([16, 1024]) + Lora_B: torch.Size([1024, 16])
Layer 3: W: torch.Size([1024, 1024]) + Lora_A: torch.Size([16, 1024]) + Lora_B: torch.Size([1024, 16])
Layer 4: W: torch.Size([1024, 2048]) + Lora_A: torch.Size([16, 2048]) + Lora_B: torch.Size([1024, 16])
Layer 5: W: torch.Size([2048, 1024]) + Lora_A: torch.Size([16, 1024]) + Lora_B: torch.Size([2048, 16])
Layer 6: W: torch.Size([1024, 1024]) + Lora_A: torch.Size([16, 1024]) + Lora_B: torch.Size([1024, 16])
Layer 7: W: torch.Size([1024, 1024]) + Lora_A: torch.Size([16, 1024]) + Lora_B: torch.Size([1024, 16])
Layer 8: W: torch.Size([1024, 2048]) + Lora_A: torch.Size([16, 2048]) + Lora_B: torch.Size([1024, 16])
Layer 9: W: torch.Size([2048, 1024]) + Lora_A: torch.Size([16, 1024]) + Lora_B: torch.Size([2048, 16])
Layer 10: W: torch.Size([1024, 1024]) + Lora_A: torch.Size([16, 1024]) + 

In [58]:
total_parameters_lora

4587520

In [57]:
total_parameters_lora / total_parameters_non_lora

0.026041666666666668

In [62]:
for name, param in model.named_parameters():
  if "lora" not in name:
    print(f"freezing {name} parameters")
    param.requires_grad = False
  else:
    pass

freezing model.embed_tokens.weight parameters
freezing model.layers.0.self_attn.q_proj.parametrizations.weight.original parameters
freezing model.layers.0.self_attn.k_proj.parametrizations.weight.original parameters
freezing model.layers.0.self_attn.v_proj.parametrizations.weight.original parameters
freezing model.layers.0.self_attn.o_proj.parametrizations.weight.original parameters
freezing model.layers.0.self_attn.q_norm.weight parameters
freezing model.layers.0.self_attn.k_norm.weight parameters
freezing model.layers.0.mlp.gate_proj.weight parameters
freezing model.layers.0.mlp.up_proj.weight parameters
freezing model.layers.0.mlp.down_proj.weight parameters
freezing model.layers.0.input_layernorm.weight parameters
freezing model.layers.0.post_attention_layernorm.weight parameters
freezing model.layers.1.self_attn.q_proj.parametrizations.weight.original parameters
freezing model.layers.1.self_attn.k_proj.parametrizations.weight.original parameters
freezing model.layers.1.self_attn.v

In [82]:
num_trainable_params = 0
num_nontrainable_params = 0
for name, param in model.named_parameters():
  if param.requires_grad == True:
    num_trainable_params += param.data.nelement()
  else:
    num_nontrainable_params += param.data.nelement()


In [83]:
print("num_trainable params : ", num_trainable_params)
print("num_nontrainable_params params : ", num_nontrainable_params)


num_trainable params :  4587520
num_nontrainable_params params :  596049920


In [74]:
from datasets import load_dataset, splits
numina_math = load_dataset("AI-MO/NuminaMath-CoT", split="train[:12000]")
def refine_prompt(example):
  return "Think step by step and solve the problem and write the final result on a new line as:\n" + example
def format_for_completion_loss(example):
  # Adjust this split logic to match whatever your actual separator is
  prompt = refine_prompt(example["problem"])

  return {
      "prompt": prompt + "\n", ### Assistant:", # Everything up to and including the tag
      "completion": "Let me think step by step: \n" + example["solution"]
  }
numina_math_dataset = numina_math.map(format_for_completion_loss)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/train-00001-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/train-00002-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/train-00003-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/train-00004-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/166k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/859494 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

In [75]:
train_dataset = numina_math_dataset.shuffle(seed=77)

In [ ]:
from trl import SFTConfig, SFTTrainer
import torch

# This is the "new API" the error is asking for!
# It globally enables TF32 in a way that torch.compile understands.
torch.set_float32_matmul_precision('high')
training_args = SFTConfig(
    output_dir="./translation",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    auto_find_batch_size=True,
    group_by_length=True,
    learning_rate=2e-4,
    logging_steps=20,
    max_steps=1500,
    bf16=True,
    save_strategy="steps",
    save_steps=500,
    tf32=True,            # <-- Now it is safe to let HF handle TF32
    optim="adamw_torch_fused",
    # --- THE NATIVE SFT FLAGS ---
    completion_only_loss=True,  # This will now work perfectly
    # dataset_text_field="text" <--- DELETE THIS.
)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    # data_collator=... <--- You don't need this anymore.
)

In [85]:

num_trainable_params = 0
num_nontrainable_params = 0
for name, param in trainer.model.named_parameters():
  if param.requires_grad == True:
    num_trainable_params += param.data.nelement()
  else:
    num_nontrainable_params += param.data.nelement()

num_trainable_params, num_nontrainable_params

(4587520, 596049920)

In [81]:
trainer.train()

Step,Training Loss
20,0.639006
40,0.585710
60,0.570433
80,0.580064
100,0.536558
120,0.691964
140,0.613851
160,0.579596
180,0.633380
200,0.528028


KeyboardInterrupt: 